In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics import classification_report,roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

In [3]:
hr=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\HR_comma_sep.csv")
hr

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [5]:
le=LabelEncoder()
hr['left']=le.fit_transform(hr['left'])
X,y=hr.drop('left',axis=1), hr['left']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=26,stratify=hr['left'])

In [6]:
# knn=KNeighborClassifier(n_neighbors=6)
# knn.fit(X_train,y_train)
# y_pred=knn.predict(X_test)

In [7]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
transf=ColumnTransformer(transformers=[("OHE",ohe, make_column_selector
                                        (dtype_include=object))],remainder="passthrough",verbose_feature_names_out=False).set_output(transform="pandas")
X_trn_ohe=transf.fit_transform(X_train)
X_tst_ohe=transf.transform(X_test)

In [9]:
knn=KNeighborsClassifier(n_neighbors=6)
knn.fit(X_trn_ohe,y_train)
y_pred=knn.predict(X_tst_ohe)
y_pred_prob=knn.predict_proba(X_tst_ohe)
roc_auc_score(y_test,y_pred_prob[:,1])

0.9725603224830541

In [21]:
Ks=[1,2,3,4,5,6,7,8,9,10]
scores=[]
for k in Ks:
    knn=KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_trn_ohe,y_train)
    y_pred_prob=knn.predict_proba(X_tst_ohe)
    scores.append([k,roc_auc_score(y_test,y_pred_prob[:,1])])
df_scores=pd.DataFrame(scores,columns=['K','score'])
df_scores.sort_values('score',ascending=False)

,K,score
7,8,0.973812
8,9,0.973488
6,7,0.973273
9,10,0.972681
5,6,0.972560
4,5,0.971581
3,4,0.970761
2,3,0.967524
1,2,0.963435
0,1,0.951201
